# Echo — collaborative filtering with matrix factorization

**Task 7, Data Science track.** This notebook reproduces the MovieLens experiment and demonstrates top-five recommendations for three anonymous users. The data are downloaded from the [GroupLens MovieLens latest-small release](https://grouplens.org/datasets/movielens/latest/) when absent.

Run from the repository root after `pip install -r requirements.txt`. We split each user’s ratings into train, validation, and test; validation selects SVD rank and shrinkage; test is evaluated once after selection.

In [ ]:
import sys
from pathlib import Path
root = Path.cwd()
if not (root / "src/train.py").exists():
    root = root.parent
sys.path.insert(0, str(root))
from src.train import load_data, split_ratings, run
import json
import matplotlib.pyplot as plt


## Data and split

Each row is an explicit 0.5–5 star rating. A per-user randomized split with a fixed seed preserves training history for every user. We do not use test ratings to choose hyperparameters.

In [ ]:
ratings, movies = load_data()
train, validation, test = split_ratings(ratings)
print(f"Ratings: {len(ratings):,}; users: {ratings.userId.nunique():,}; movies: {len(movies):,}")
print(f"Training: {len(train):,}; validation: {len(validation):,}; test: {len(test):,}")
display(ratings.head())


## Baseline, SVD, and held-out evaluation

The implementation fits regularized viewer and movie biases. It applies truncated sparse SVD to the residuals of observed training ratings, reconstructs predicted ratings, and clips to the valid 0.5–5 range. Validation RMSE selects factor count and shrinkage. The final model refits on train + validation and is scored once on test.

In [ ]:
metrics = run()  # Also writes public/metrics.json and public/recommendations.json
display({key: metrics[key] for key in ("selected", "test_baseline", "test_model")})


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4))
labels = ["Bias-only baseline", "Matrix factorization"]
values = [metrics["test_baseline"]["rmse"], metrics["test_model"]["rmse"]]
bars = ax.bar(labels, values, color=["#738095", "#7b9e4f"])
ax.bar_label(bars, fmt="%.4f", padding=3)
ax.set(ylabel="Test RMSE (lower is better)", ylim=(0, max(values) * 1.16), title="Unseen rating prediction error")
fig.tight_layout()
plt.show()


## Top-five predictions for three users

The deployment snapshot excludes movies each user rated during model fitting. These scores are predicted ratings, not watch probabilities or proof of real-world engagement.

In [ ]:
profiles = json.loads((root / "public/recommendations.json").read_text())
for user_id in (1, 42, 100):
    print(f"\nUser {user_id} ({profiles[str(user_id)]['ratedCount']} training ratings)")
    for rec in profiles[str(user_id)]["recommendations"]:
        print(f"  {rec['score']:.2f}  {rec['title']}")
